In [ ]:
import pandas as pd
import numpy as np

# 1. Load the data
df = pd.read_csv("rfm_data.csv")
df['PurchaseDate'] = pd.to_datetime(df['PurchaseDate'])

# 2. Re-calculate everything including Segment
reference_date = df['PurchaseDate'].max() + pd.Timedelta(days=1)
rfm = df.groupby('CustomerID').agg({
    'PurchaseDate': lambda x: (reference_date - x.max()).days,
    'OrderID': 'count',
    'TransactionAmount': 'sum'
}).reset_index()
rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']

# Scores
rfm['R_Score'] = pd.cut(rfm['Recency'], bins=[-1, 14, 31, 44, 100], labels=[4, 3, 2, 1]).astype(int)
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), q=4, labels=[1, 2, 3, 4]).astype(int)
rfm['M_Score'] = pd.qcut(rfm['Monetary'], q=4, labels=[1, 2, 3, 4]).astype(int)

# Segmenting logic
def assign_segment(row):
    if row['R_Score'] >= 3 and row['F_Score'] >= 3:
        return 'Champions'
    elif row['R_Score'] >= 3 and row['F_Score'] < 3:
        return 'Active Customers'
    elif row['R_Score'] < 3 and row['F_Score'] >= 3:
        return 'At Risk'
    else:
        return 'Lost Customers'

rfm['Segment'] = rfm.apply(assign_segment, axis=1)

# 3. Save the final file
rfm.to_csv("full_rfm_analysis_final.csv", index=False)
print("File 'full_rfm_analysis_final.csv' created with Segment column included.")

File 'full_rfm_analysis_final.csv' created with Segment column included.
